In [ ]:
import sys
sys.path.insert(0, "/home/claude")
from datetime import datetime
import pandas as pd
import networkx as nx
from visualize_ean import plot_ean, draw_ean
from build_ean import build_ean, add_headway_arcs, propagate, enrich_trip_data_with_boundaries
import headway_integration_copy as hi
import numpy as np
from collections import defaultdict


In [ ]:
def reassign_trip_ids_by_departure(trip_data, first_station="ME", second_station="MAL", save_path=None):
    entries_first = []
    entries_second = []
    entries_other = []

    for old_id, stops in trip_data.items():
        if not stops:
            continue
        first_stop = stops[0][0]
        # prefer explicit departure time at index 2, fallback to index 1
        dep_time = None
        if len(stops[0]) > 2 and stops[0][2] is not None:
            dep_time = stops[0][2]
        elif len(stops[0]) > 1:
            dep_time = stops[0][1]
        else:
            dep_time = pd.NaT

        record = (old_id, dep_time)
        if first_stop == first_station:
            entries_first.append(record)
        elif first_stop == second_station:
            entries_second.append(record)
        else:
            entries_other.append(record)

    # sort by departure time (NaT will go last)
    def _sort_key(x): return (pd.Timestamp.max if pd.isna(x[1]) else x[1])
    entries_first.sort(key=_sort_key)
    entries_second.sort(key=_sort_key)
    entries_other.sort(key=_sort_key)

    ordered_old_ids = [r[0] for r in (entries_first + entries_second + entries_other)]

    new_trip_data = {}
    for new_id, old_id in enumerate(ordered_old_ids, start=1):
        new_trip_data[new_id] = trip_data[old_id]

    if save_path:
        np.save(save_path, new_trip_data, allow_pickle=True)

    return new_trip_data, ordered_old_ids

# Example usage:
# new_trip_data, ordered_old_ids = reassign_trip_ids_by_departure(trip_data, "ME", "MAL", save_path="inputs2/trip_data_reassigned.npy")
# selected_trips_sorted = np.array(list(new_trip_data.keys()))

In [ ]:
scenario = "1a"

SCENARIOS = {
    "0": {"base", "existing"},
    "1a": {"base", "existing", "dt_me_lag", "dt_tel_pla", "dt_natkomp_sta", "dt_sta_cia", "dt_cold_sblLac", "dt_lasa_oris"},
    "1b": {"base", "existing"},
    "2a": {"base", "existing"},
    "all": None,   # special case
}

ACTIVE_SCENARIO = SCENARIOS[scenario]

def load_network_csv(path, index_col):
    df = pd.read_csv(path, sep=";", decimal=",", index_col=index_col)

    if scenario == "all":
        if "edge_type" in df.columns:
            return df[df["edge_type"] != "connecting"]
        return df

    return df[df["scenario"].fillna("").apply(lambda s: bool(set(map(str.strip, s.split(","))) & ACTIVE_SCENARIO))]

In [ ]:
nodesDf = load_network_csv(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\SimulatedAnnealing\network_with_altitude\double_track_network\nodes_double.csv","node_id")
edgesDf = load_network_csv(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\SimulatedAnnealing\network_with_altitude\double_track_network\edges_double.csv","edge_id")
edgesDf["length"] = (edgesDf["node_to"].map(nodesDf["pk_rel"])- edgesDf["node_from"].map(nodesDf["pk_rel"])).abs()
#trip_data = np.load(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\plottingRailML\230215 FBS Fpl 2026-NB-1443983-3.npy", allow_pickle=True).item()
trip_data = np.load(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\plottingRailML\RIdottoStamm6.npy", allow_pickle=True).item()
#trip_data = np.load(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\SimulatedAnnealing\MinimumHeadways\retry\data\trip_data_17.npy", allow_pickle=True).item()
headway_dict = np.load(fr"C:\Users\LeoC\VSCodes\optimizationVinschgau\SimulatedAnnealing\Conflicts\all_min_headways{scenario}.npy", allow_pickle=True).item()
selected_trips_sorted = np.array(list(trip_data.keys()))

In [ ]:
trip_data = {
    train_id: [
        (station, arr_dt, dep_dt, True)
        for station, arr_dt, dep_dt in stops
    ]
    for train_id, stops in trip_data.items()
}

In [ ]:
for t in trip_data:
    print((trip_data[t]))

In [ ]:
trip_data, ordered_old_ids=reassign_trip_ids_by_departure(trip_data)

In [ ]:
def get_routing_node(node, direction):
    """
    For trains towards Malles, use station side nodes when available.
    For trains towards Merano, always use main station nodes.
    """

    if direction == "malles":

        side_node = f"{node}_side"

        if side_node in nodesDf.index:
            return side_node

    return node


# --------------------------------------------------
# Route builder
# --------------------------------------------------

def build_route(nodesDf, edgesDf, start, end):
    successors = defaultdict(list)

    for _, edge in edgesDf.iterrows():
        successors[edge["node_from"]].append(edge["node_to"])
        successors[edge["node_to"]].append(edge["node_from"])
    direction = ("malles" if nodesDf.loc[start, "pk_rel"] <= nodesDf.loc[end, "pk_rel"] else "merano")

    route_start = get_routing_node(start, direction)
    route_end = get_routing_node(end, direction)

    route = [route_start]

    current = route_start
    visited = {route_start}

    while current != route_end:

        pk_current = nodesDf.loc[current, "pk_rel"]

        # only move towards destination
        candidates = [
            n for n in successors[current]
            if (
                nodesDf.loc[n, "pk_rel"] > pk_current
                if direction == "malles"
                else nodesDf.loc[n, "pk_rel"] < pk_current
            )
        ]

        if not candidates:
            raise ValueError(
                f"No valid successor from {current} towards {route_end}"
            )

        # at branching points choose preferred track
        if len(candidates) > 1:

            preferred = [
                n for n in candidates
                if (
                    nodesDf.loc[n, "y"] != 0
                    if direction == "malles"
                    else nodesDf.loc[n, "y"] == 0
                )
            ]

            if preferred:
                candidates = preferred

        next_node = candidates[0]

        if next_node in visited:
            raise ValueError(
                f"Loop detected: {current} -> {next_node}"
            )

        route.append(next_node)
        visited.add(next_node)

        current = next_node

    return route


# --------------------------------------------------
# Build routing dictionary
# --------------------------------------------------

routing_results = {}

for stop_col in ["stop_slow", "stop_fast"]:

    stops = nodesDf.index[(nodesDf[stop_col] == 1) & (nodesDf["y"] == 0)].tolist()

    for start, end in zip(stops[:-1], stops[1:]):

        routing_results[(start, end)] = build_route(nodesDf, edgesDf, start, end)
        routing_results[(end, start)] = build_route(nodesDf, edgesDf, end, start)

In [ ]:
routes = {}
for trip_id, trip in trip_data.items():
    start, end = trip[0][0], trip[-1][0]
    routes[trip_id] = build_route(nodesDf, edgesDf, start, end)

In [ ]:
from copy import deepcopy

def add_side_nodes_to_trip_data(trip_data, nodesDf):
    """
    Return a copy of trip_data where terminal nodes are replaced by their
    '_side' counterpart, but only for trains running from Merano to Malles
    (i.e. decreasing pk_rel).

    A stop 'X' is replaced by 'X_side' only if:
      - 'X_side' exists in nodesDf, and
      - the train direction is Merano -> Malles.
    """
    trip_data_sides = deepcopy(trip_data)

    available_nodes = set(nodesDf.index)

    for train_id, stops in trip_data_sides.items():

        # Need at least two stops to infer direction
        if len(stops) < 2:
            continue

        pk0 = nodesDf.loc[stops[0][0], "pk_rel"]
        pk1 = nodesDf.loc[stops[1][0], "pk_rel"]

        # Merano -> Malles corresponds to decreasing pk_rel
        merano_to_malles = pk1 > pk0

        if not merano_to_malles:
            continue

        for i, (node, arr, dep, is_stop) in enumerate(stops):

            side_node = f"{node}_side"

            if side_node in available_nodes:
                stops[i] = (side_node, arr, dep, True)

    return trip_data_sides

In [ ]:
trip_data_sides = add_side_nodes_to_trip_data(trip_data, nodesDf)

In [ ]:
IG = hi.build_infra_graph(edgesDf)
chains, boundary_nodes = hi.extract_chains(IG, nodesDf)
print("Chains:", chains)
print("Boundary Nodes:", boundary_nodes)
trip_data_enriched = enrich_trip_data_with_boundaries(trip_data_sides, routes, nodesDf, chains)

constraints, skipped = hi.assemble_headway_constraints(trip_data_sides, trip_data_enriched, routes, nodesDf, chains, headway_dict)
print("Constraints:", constraints)
print("Skipped:", skipped)
#assert len(constraints) == 1, f"expected 1 headway arc between the 2 trains, got {len(constraints)}"

G_scheduled = build_ean(trip_data_enriched)
G_scheduled = add_headway_arcs(G_scheduled, constraints)
assert nx.is_directed_acyclic_graph(G_scheduled), "graph must stay a DAG"

realized = propagate(G_scheduled, {})

In [ ]:
running_edges = [(u, v) for u, v, data in G_scheduled.edges(data=True) if data["kind"] == "running"]

#perturbations = [{},{edge: 60},{edge: 120},{edge: 300}]
perturbations = [{}]

realized_graphs = []

for p in perturbations:

    G_realized = propagate(G_scheduled, p)

    realized_graphs.append(G_realized)


fig, ax = plot_ean(G_scheduled, nodesDf, edgesDf, title="Scheduled vs realized")

for G_realized in realized_graphs:
    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)

In [ ]:
def compute_train_durations(G):
    """
    Returns
    -------
    dict
        {train_id: duration (timedelta)}
    """
    durations = {}
    trains = sorted({data["train"] for _, data in G.nodes(data=True) if "train" in data})

    for train in trains:
        events = [data for _, data in G.nodes(data=True) if data.get("train") == train]
        dep = min(e["time"] for e in events if e["event"] == "dep")
        arr = max(e["time"] for e in events if e["event"] == "arr")
        durations[train] = arr - dep
        
    return durations

In [ ]:
def compare_to_schedule(G_scheduled, G_realized):
    scheduled = compute_train_durations(G_scheduled)
    realized = compute_train_durations(G_realized)
    report = {}
    for train in scheduled:
        report[train] = {"scheduled": scheduled[train], "realized": realized[train], "extra_time": realized[train] - scheduled[train]}
    return report

In [ ]:
scheduled = compute_train_durations(G_scheduled)

for i, (perturbation, G_realized) in enumerate(zip(perturbations, realized_graphs)):
    realized = compute_train_durations(G_realized)
    primary = sum(perturbation.values()) if perturbation else 0
    print(f"\nScenario {i}  (primary delay = {primary:.0f} s)")
    print("-" * 45)
    secondary = 0
    for train in sorted(scheduled):
        extra = realized[train] - scheduled[train]
        # primary delay belongs to the perturbed train only
        propagated = extra - primary if train == 1 else extra
        secondary += max(0, propagated)
        print(
            f"Train {train:>2}: "
            f"scheduled={scheduled[train]:6.0f}s  "
            f"realized={realized[train]:6.0f}s  "
            f"Δ={extra:5.0f}s"
        )

    print(f"\nPrimary delay   : {primary:.0f} s")
    print(f"Secondary delay : {secondary:.0f} s")